In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import pandas as pd

books = pd.read_csv("./books_cleaned.csv")

In [5]:
books["tagged_description"].head()

0    9780002005883 A NOVEL THAT READERS and critics...
1    9780002261982 A new 'Christie for Christmas' -...
2    9780006178736 A memorable, mesmerizing heroine...
3    9780006280897 Lewis' work on the nature of lov...
4    9780006280934 "In The Problem of Pain, C.S. Le...
Name: tagged_description, dtype: object

In [5]:
books["tagged_description"].to_csv("./tagged_description.txt",
                                   index = False,
                                   header = False)

In [6]:
raw_documents = TextLoader("./tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=1, chunk_overlap=0, separator="\n")

documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1170, which is longer than the specified 1
Created a chunk of size 1216, which is longer than the specified 1
Created a chunk of size 375, which is longer than the specified 1
Created a chunk of size 311, which is longer than the specified 1
Created a chunk of size 483, which is longer than the specified 1
Created a chunk of size 484, which is longer than the specified 1
Created a chunk of size 962, which is longer than the specified 1
Created a chunk of size 188, which is longer than the specified 1
Created a chunk of size 845, which is longer than the specified 1


Created a chunk of size 296, which is longer than the specified 1
Created a chunk of size 197, which is longer than the specified 1
Created a chunk of size 881, which is longer than the specified 1
Created a chunk of size 1090, which is longer than the specified 1
Created a chunk of size 1191, which is longer than the specified 1
Created a chunk of size 306, which is longer than the specified 1
Created a chunk of size 270, which is longer than the specified 1
Created a chunk of size 213, which is longer than the specified 1
Created a chunk of size 216, which is longer than the specified 1
Created a chunk of size 515, which is longer than the specified 1
Created a chunk of size 754, which is longer than the specified 1
Created a chunk of size 390, which is longer than the specified 1
Created a chunk of size 265, which is longer than the specified 1
Created a chunk of size 253, which is longer than the specified 1
Created a chunk of size 308, which is longer than the specified 1
Created 

In [7]:
documents[0].page_content

'"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the wor

In [8]:
# Using a free langchain embedding instead of OpenAI paid service
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-l6-v2')


db_books = Chroma.from_documents(
    documents,
    embedding=embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
query = "A book teaching young adults on about health"

docs = db_books.similarity_search(query, k = 5)

docs_isbn13 = [ int(doc.page_content.strip('"').split()[0]) for doc in docs ]
docs_isbn13

relevant_books = books[books.isbn13.isin(docs_isbn13)]
relevant_books[['isbn13', 'description']]

,isbn13,description
1652,9780374528539,"A lighthearted but insightful guide to raising adolescent children shows parents how to deal with teenagers living in a faster-paced, less morally certain world than the one they knew. Original. 50,000 first printing."
2925,9780618479030,"The best-selling books of Andrew Weil, ""the guru of alternative medicine,"" (San Francisco Examiner) offer a comprehensive blend of traditional and alternative methods that help to achieve better health in the modern world. Natural Health, Natural Medicine is a comprehensive resource for everything you need to know to maintain optimum health and treat common ailments. This landmark book incorporates Dr. Weil’s theories of preventive health maintenance and alternative healing into one extremely useful and readable reference, featuring general diet and nutrition information as well as simple recipes, answers to readers’ most pressing questions, a catalogue of home remedies, invaluable resources, and hundreds of practical tips. This edition includes up-to-the-minute scientific findings and has been expanded to provide trustworthy advice about low-carb diets, hormone replacement therapy, Alzheimer’s, attention deficit disorder, reflux disease, autism, type 2 diabetes, erectile dysfunction, the flu, and much more."
4252,9780936197449,"A survival guide for parents helps beleagured adults navigate their childrens' teen years, with advice on how to turn adolescents into strong, confident, productive adults."
4421,9781401211349,"Presents a practical guide that addresses many of the tough issues teens and parents face, and offers advice to maintaining a positive dialogue that fostered mutual respect."
4713,9781576839546,"Cline and Fay offer advice to help parents raise kids who are self-confident, motivated, and ready for the world by teaching them responsibility and the logic of life, thereby giving them the opportunity to solve their own problems from the earliest possible age."


In [49]:
def retrieve_semantic_recommendation(
        query: str,
        top_k: int = 8
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=top_k)

    books_list = []
    for i in range(len(recs)):
        books_list.append(int(recs[i].page_content.strip('"').split()[0]))
    
    return books_list 

In [50]:
retrieved_isbn13 = retrieve_semantic_recommendation("Science and Technology", top_k=3)

relevant_books = books[books.isbn13.isin(retrieved_isbn13)]
relevant_books[['isbn13', 'description']]

,isbn13,description
2887,9780618056736,"Offers an assessment of what science is, how it feeds the human appetite for wonder, and how ""unweaving"" the mysteries of science can be even more beautiful than the mystery itself."
3247,9780713998788,"The explosion of advanced technologies, knowledge pools and resources that have connected over the planet levelling the playing field as never before, each of us is potentially an equal and competitor of the other."
3835,9780802713520,"Generations have grown up knowing that the equation E=mc2 changed the shape of our world, but never understanding what it actually means, why it was so significant, and how it informs our daily lives today—governing, as it does, everything from the atomic bomb to a television's cathode ray tube to the carbon dating of prehistoric paintings. In this book, David Bodanis writes the ""biography"" of one of the greatest scientific discoveries in history—that the realms of energy and matter are inescapably linked—and, through his skill as a writer and teacher, he turns a seemingly impenetrable theory into a dramatic human achievement and an uncommonly good story."


In [51]:
books['categories'].value_counts().reset_index()

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
...,...,...
474,Aged women,1
475,Imperialism,1
476,Human-animal relationships,1
477,Amish,1


In [52]:
books['categories'].value_counts().reset_index().query('count > 50')

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Religion,117
6,Philosophy,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [59]:
category_mapping = {
    'Fiction': 'Fiction',
    'Juveline Fiction': "Children's Fiction",
    'Biography & Autobiography': 'Nonfiction',
    'History': 'Nonfiction',
    'Literary Criticism': 'Nonfiction',
    'Philosophy': 'Nonfiction',
    'Religion': 'Nonfiction',
    'Comics & Graphic Novels': 'Fiction',
    'Drama': 'Fiction',
    'Juveline Nonfiction': "Children's Nonfiction",
    'Science': 'Nonfiction',
    'Poetry': 'Fiction'
}

In [61]:
books['simple_categories'] = books['categories'].map(category_mapping)

In [64]:
books[~(books['simple_categories'].isna())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api,"A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world has to offer. At its heart is a tale of the sacred bonds between fathers and sons, pitch-perfect in style and story, set to dazzle critics and readers alike.",2004.0,3.85,247.0,361.0,Gilead,"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world has to offer. At its heart is a tale of the sacred bonds between fathers and sons, pitch-perfect in style and story, set to dazzle critics and readers alike.",Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2TgANz74C&printsec=frontcover&img=1&zoom=1&source=gbs_api,"A memorable, mesmerizing heroine Jennifer -- brilliant, beautiful, an attorney on the way up until the Mafia's schemes win her the hatred of an implacable enemy -- and a love more destructive than hate. A dangerous, dramatic world The Dark Arena of organized crime and flashbulb lit courtrooms where ambitious prosecutors begin their climb to political power.",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine Jennifer -- brilliant, beautiful, an attorney on the way up until the Mafia's schemes win her the hatred of an implacable enemy -- and a love more destructive than hate. A dangerous, dramatic world The Dark Arena of organized crime and flashbulb lit courtrooms where ambitious prosecutors begin their climb to political power.",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0fpS9WZkC&printsec=frontcover&img=1&zoom=1&source=gbs_api,"Tricked once more by his wily half-brother, Lysaer, Lord of Light, arrives at the tiny harbor town of Merior to find that Arithon's ship yards have been abandoned and meticul